In [15]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from resources import find_tilt_error
from resources.cut_off_edges import cut_off_edges
from resources.find_rotation_error import find_rotation_error_by_profile
from resources.find_tilt_error import find_tilt_error_by_profile
from resources.rotate_image import rotate_volume
from resources.standarize_image import standardize_image

In [16]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1362
Liczba etykiet: 1362


In [ ]:
from resources.find_z import calculate_z_profile, find_z_area_center
from resources.find_y import calculate_y_profile, find_y_area_center
from resources.find_x import calculate_x_profile, find_x_area_center

volumes = []
y_labels = []
uids = []

for index, nifti_path in enumerate(nifti_paths, start=1):
    uid = nifti_path.name.removesuffix(".nii.gz")

    if uid not in labels.index:
        print(f"Brak etykiety dla {uid}")
        continue

    print(f"{index}/{len(nifti_paths)}: {uid}")

    image = nib.load(nifti_path)

    processed_image = cut_off_edges(image)

    processed_image_rotation = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )
    processed_image_rotation = cut_off_edges(processed_image_rotation, x_left=30, x_right=30, y_front=30, y_back=30, z_down=20, z_up=25)
    processed_image_rotation = standardize_image(processed_image_rotation, percent_top=0.02)
    rotation_error = find_rotation_error_by_profile(processed_image_rotation)
    tilt_error = find_tilt_error_by_profile(processed_image_rotation)
    x_profile = calculate_x_profile(processed_image_rotation.get_fdata(dtype=np.float32))
    y_profile = calculate_y_profile(processed_image_rotation.get_fdata(dtype=np.float32))
    z_profile = calculate_z_profile(processed_image_rotation.get_fdata(dtype=np.float32))

    x_centre = int(find_x_area_center(x_profile))
    y_centre = int(find_y_area_center(y_profile))
    z_centre = int(find_z_area_center(z_profile))

    centre = (
    x_centre,
    y_centre,
    z_centre,
    )

    processed_image = rotate_volume(processed_image, rotation_error, axis="x", center=centre)
    processed_image = rotate_volume(processed_image, tilt_error, axis="z",center=centre)
    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=25,
        z_up=30,
    )
    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )
    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=25,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=25,
        x_right=25,
        y_front=30,
        y_back=10,
        z_down=10,
        z_up=25,
    )
    volume = processed_image.get_fdata(dtype=np.float32)
    volumes.append(volume)
    y_labels.append(int(labels.loc[uid]))
    uids.append(uid)

In [ ]:
from collections import Counter

shape_counts = Counter(volume.shape for volume in volumes)
print(shape_counts)
X = np.stack(volumes).astype(np.float32)
y = np.asarray(y_labels, dtype=np.int64)
uids = np.asarray(uids)

print("X:", X.shape)
print("y:", y.shape)
X = X[:, np.newaxis, ...]

print(X.shape)

In [ ]:
np.savez_compressed(
    project_dir / "data" / "processed_volumes_with_rotation.npz",
    uid=uids,
    X=X.astype(np.float16),
    y=y,
)

In [17]:
data = np.load(
    project_dir / "data" / "processed_volumes_with_rotation.npz",
    allow_pickle=False,
)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
uids = data["uid"]

In [18]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
indices = np.arange(len(y))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

uids_train = uids[train_indices]
uids_test = uids[test_indices]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1089, 1, 50, 40, 35)
X_test: (273, 1, 50, 40, 35)


In [19]:
class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()
        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

In [20]:
train_dataset = VolumeDataset(X_train, y_train)
test_dataset = VolumeDataset(X_test, y_test)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

In [21]:
from torch import nn
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=8,
                out_channels=16,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [22]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Simple3DCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

print("Urządzenie:", device)

Urządzenie: cpu


In [23]:
import copy

best_validation_loss = float("inf")
best_model_state = None

epochs = 25

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for volumes_batch, labels_batch in train_loader:
        volumes_batch = volumes_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        logits = model(volumes_batch)
        loss = criterion(logits, labels_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(labels_batch)

    train_loss /= len(train_dataset)

    model.eval()
    validation_loss = 0.0

    with torch.no_grad():
        for volumes_batch, labels_batch in validation_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            validation_loss += loss.item() * len(labels_batch)

    validation_loss /= len(test_dataset)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"train: {train_loss:.4f} | "
        f"validation: {validation_loss:.4f}"
    )

model.load_state_dict(best_model_state)

print("Najlepszy validation loss:", best_validation_loss)

Epoka 1/25 | train: 0.6838 | validation: 0.7189
Epoka 2/25 | train: 0.6814 | validation: 0.6544
Epoka 3/25 | train: 0.6451 | validation: 0.6846
Epoka 4/25 | train: 0.6667 | validation: 0.6608
Epoka 5/25 | train: 0.6556 | validation: 0.6456
Epoka 6/25 | train: 0.6396 | validation: 0.6461
Epoka 7/25 | train: 0.6185 | validation: 0.5843
Epoka 8/25 | train: 0.5769 | validation: 0.5726
Epoka 9/25 | train: 0.5803 | validation: 0.5952
Epoka 10/25 | train: 0.5158 | validation: 0.4503
Epoka 11/25 | train: 0.4747 | validation: 0.4275
Epoka 12/25 | train: 0.4657 | validation: 0.4670
Epoka 13/25 | train: 0.4638 | validation: 0.4734
Epoka 14/25 | train: 0.4672 | validation: 0.4237
Epoka 15/25 | train: 0.4492 | validation: 0.4013
Epoka 16/25 | train: 0.4468 | validation: 0.4233
Epoka 17/25 | train: 0.4391 | validation: 0.3924
Epoka 18/25 | train: 0.4397 | validation: 0.3883
Epoka 19/25 | train: 0.4449 | validation: 0.4114
Epoka 20/25 | train: 0.4382 | validation: 0.3830
Epoka 21/25 | train: 0.4435 |

In [12]:
from pathlib import Path
from datetime import datetime
import torch


models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

model_path = models_dir / f"simple_3d_cnn_rotation_{timestamp}_{best_validation_loss}.pth"

torch.save(
    model.state_dict(),
    model_path,
)

print(f"Zapisano model: {model_path} {best_validation_loss}")

Zapisano model: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\models\simple_3d_cnn_2026-09-05_20-59-45_0.24123360516824127.pth 0.24123360516824127


In [ ]:
from notebooks.EDA4.calculate_metrics import calculate_metrics
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier

best_f1_score_test = -1
lowest_f1_score_test = 10
for max_depth in range(1,20):
    for min_samples_leaf in range(2,20):
        name = f'Tree_depth:{max_depth}_min_sample_leaf:{min_samples_leaf}'
        model_dtc = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        model_dtc.fit(X_train, y_train)
        prediction_train = model_dtc.predict(X_train)
        prediction_test = model_dtc.predict(X_test)
        f1_score_train = f1_score(y_train, prediction_train)
        f1_score_test = f1_score(y_test, prediction_test)
        print(f'for : {name}')
        print('F1 train:', f1_score_train)
        print('F1 test :', f1_score_test)
        if f1_score_test > best_f1_score_test:
            best_f1_score_test = f1_score_test
            best_name = name
        if f1_score_test < lowest_f1_score_test:
            lowest_f1_score_test = f1_score_test
            lowest_name = name

print('for:  ',best_name,'\nBest score F1 test = ', best_f1_score_test)
print('for:  ',lowest_name,'\nLowest score F1 test = ', lowest_f1_score_test)

print('Best score')
model_dtc = DecisionTreeClassifier(max_depth=2,min_samples_leaf=2)
model_dtc.fit(X_train, y_train)
# plot_classification_surface(X_plot=X_train, y_plot=y_train, trained_model=model_knn)
calculate_metrics(model_dtc, 'DecisionTree_best', X_test, y_test)
print('Lowest score')
model_dtc = DecisionTreeClassifier(max_depth=10,min_samples_leaf=10)
model_dtc.fit(X_train, y_train)
# plot_classification_surface(X_plot=X_train_standardized, y_plot=y_train, trained_model=model_knn)
calculate_metrics(model_dtc, 'DecisionTree_best', X_test, y_test)